In [1]:
# Load environment and suppress warnings
import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import required components and initialize deterministic model
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

model = ChatOpenAI(model="gpt-5-mini", temperature=0)
parser = StrOutputParser()

In [3]:
# Define a sample request (context plus metadata fields)
request = {
    "ticket_text": "Users cannot log in after the latest deployment.",
    "request_id": "REQ-1042",
    "customer_tier": "Gold"
}

In [4]:
# Step 1 prompt: extract a short issue phrase
extract_issue_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Extract a short issue phrase (2 to 6 words). Return only the phrase."),
        ("human", "Ticket: {ticket_text}")
    ]
)

extract_issue_chain = extract_issue_prompt | model | parser

# Step 1 function: add extracted issue to the context
def add_extracted_issue(ctx: Dict[str, Any]) -> Dict[str, Any]:
    
    extracted_issue = extract_issue_chain.invoke(
        {"ticket_text": ctx["ticket_text"]}
    ).strip()

    updated = dict(ctx)
    updated["extracted_issue"] = extracted_issue

    return updated

In [5]:
# Step 2 prompt: create one annotation based on context
annotate_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Write one short annotation about the issue."),
        ("human", "Ticket: {ticket_text}\nIssue: {extracted_issue}\nCustomer tier: {customer_tier}")
    ]
)

annotate_chain = annotate_prompt | model | parser

# Step 2 function: add annotation to the context
def add_annotation(ctx: Dict[str, Any]) -> Dict[str, Any]:
    annotation = annotate_chain.invoke(
        {
            "ticket_text": ctx["ticket_text"],
            "extracted_issue": ctx["extracted_issue"],
            "customer_tier": ctx["customer_tier"],
        }
    ).strip()
    updated = dict(ctx)
    updated["annotation"] = annotation
    return updated

In [6]:
# Step 3 prompt: generate final recommended action
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Return one recommended support action."),
        ("human", "Request ID: {request_id}\nIssue: {extracted_issue}\nNote: {annotation}")
    ]
)

final_chain = final_prompt | model | parser

In [7]:
# Compose the workflow: context flows through steps and is enriched along the way
workflow = (
    RunnableLambda(add_extracted_issue)
    | RunnableLambda(add_annotation)
    | final_chain
)

# Execute the workflow
print("=== INPUT CONTEXT ===")
print(request)

final_output = workflow.invoke(request)

print("\n=== FINAL OUTPUT ===")
print(final_output)

=== INPUT CONTEXT ===
{'ticket_text': 'Users cannot log in after the latest deployment.', 'request_id': 'REQ-1042', 'customer_tier': 'Gold'}

=== FINAL OUTPUT ===
Immediate rollback: revert the latest deployment (at least the authentication-related services or the entire release if auth is entangled) to the last known-good version immediately, put the system into a brief maintenance state to prevent further failed logins, and preserve all current logs/traces/artifacts for post-rollback investigation.


In [8]:
# Inspect intermediate results to see how context evolves

after_issue = add_extracted_issue(request)

print("\n=== AFTER ISSUE EXTRACTION ===")
print({
    "ticket_text": after_issue["ticket_text"],
    "extracted_issue": after_issue["extracted_issue"]
})

after_annotation = add_annotation(after_issue)

print("\n=== AFTER ANNOTATION STEP ===")
print({
    "extracted_issue": after_annotation["extracted_issue"],
    "annotation": after_annotation["annotation"]
})


=== AFTER ISSUE EXTRACTION ===
{'ticket_text': 'Users cannot log in after the latest deployment.', 'extracted_issue': 'Login failures after deployment'}

=== AFTER ANNOTATION STEP ===
{'extracted_issue': 'Login failures after deployment', 'annotation': 'Post-deployment authentication regression causing login failures for Gold-tier users — urgent investigation and rollback/check of auth services required.'}
